<a href="https://colab.research.google.com/github/csbell-vu/in-class-notebooks/blob/main/Lab_Transfer_Learning_and_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 5690 · Lab · Transfer Learning and Training Basics

> DS 5690 - Generative AI in Theory and Practice  
> Dr. Charreau Bell  
> Fall 2026  

> **Transfer learning, batching, and KV caching**

In this session, we'll explore some practicals around transformer model training that we've only lightly covered in class. We'll also do a few experiments to prove true that is otherwise somewhat unbelievable without evidence.

**Make sure to change to a GPU runtime before starting.**



## 1. Train Your Own Tool Router for Agents with Transfer Learning

Modern day agents perform autonomous, goal-directed actions, which often requires selecting tools to complete a task. One way to do this is through using a routing model, which reads a request and predicts which tool *could* handle it.

In this section, we will employ **transfer learning** to train a LLM to identify which action to take/tool to use based on instructions from the user.

We will train an **encoder-only** model ([DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased)) and a similarly sized **decoder-only** model ([DistilGPT2](https://huggingface.co/distilbert/distilgpt2)) - both roughly 65–85M parameters - on the same labels and splits, and compare which one does better. There is _much_ more rigor we would need to apply to this analysis for it to be reasonable, but our investigation should be directionally correct.

The **dataset** that we use is [CLINC150](https://huggingface.co/datasets/clinc/clinc_oos), using five supported actions/tools plus `oos` (out of scope).


In [ ]:
%pip -q install "transformers>=4.56,<5" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn pandas

In [ ]:
import pandas as pd
import torch
from datasets import concatenate_datasets, load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification,
                          AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments,
                          set_seed,)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

#### Prep the Data

Here, we just groom the data a little bit to get it in the shape that we want. Though there are 100-something different tool routes (classes) that we could classify to in this dataset, we're just going to select 5, shown in the `routes` variable below.

In [ ]:
# These are possible agents that the agent might need to take
routes = ["weather", "calendar", "calculator", "reminder", "play_music", "oos"]

In [ ]:
# Download the dataset
raw = load_dataset("clinc/clinc_oos", "small")

# Set up the classification labels (get names)
label_column = "intent" if "intent" in raw["train"].column_names else "label"
label_names = raw["train"].features[label_column].names

# Get the IDs (for classification - we have to output numbers)
route_ids = [label_names.index(route) for route in routes]

In [ ]:
# view content of dataset in train
raw['train']

Now, we will just grab a subset of the data from the small dataset. We'll get 20-50 examples of each route type for each of the datasets.

In [ ]:
# Get 50, 20, or 30 samples of each class for each data split
split_samples = {'train':50, 'validation':20, 'test':30}

# Create ds dictionary
split_ds = {}

# Iterate and collect sample_count rows of each action type for each ds
for split, sample_count in split_samples.items():
    split_ds[split] = concatenate_datasets([
        raw[split].filter(lambda row: row[label_column] == label_id).select(range(sample_count))
        for label_id in route_ids
    ]).shuffle(seed=42)

# Create final ds
ds = DatasetDict(split_ds)

# Add label for each row
ds = ds.map(lambda row: {"label": route_ids.index(row[label_column])})

Let's see our full downsampled train, validation, and test datasets.

In [ ]:
# View number of examples in each dataset
ds

In [ ]:
# View a few examples
pd.DataFrame(ds['train'].select(range(5))[:])

### Setup shared training parameters
Now, we'll set all the training parameters that are shared to be in one place.

In [ ]:
# function to tokenize the input text based on the tokenizer
def tokenize(tokenizer, batch):
  inputs = tokenizer(batch["text"], truncation=True, max_length=64)
  return inputs

# operator to transform classification label IDs to their text label
id2label = dict(enumerate(routes))

# operator to transform class text labels into IDs
label2id = {route: i for i, route in enumerate(routes)}

# number of labels
num_labels = len(routes)

# shared model training arguments
shared_training_arguments = {
    "num_train_epochs": 3,
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "eval_strategy": "epoch",
    "save_strategy": "no",
    "logging_strategy": "epoch",
    "report_to": "none",
    "seed": 42,
    "fp16": torch.cuda.is_available()
}

Now, let's leverage transfer learning and fine-tune the encoder.

### A. Fine-tune the encoder

The pretrained encoder supplies representations; its new classification layer maps a request to one of our six routes. We update **both** the encoder and classification head.


In [ ]:
# get tokenizer
encoder_name = "distilbert/distilbert-base-uncased"
encoder_tokenizer = AutoTokenizer.from_pretrained(encoder_name)

# prepare the text inputs and labels with the tokenizer
encoder_ds = ds.map(lambda batch: tokenize(encoder_tokenizer, batch), batched=True)

Setup and do the training...

In [ ]:
# setup classification head, removing BERT MLM head
encoder_model = AutoModelForSequenceClassification.from_pretrained(
    encoder_name, num_labels=num_labels, id2label=id2label,
    label2id=label2id
)

# prepare the training arguments
encoder_args = TrainingArguments(output_dir="/content/router-encoder",
                                 **shared_training_arguments)

# setup the infrastructure for training
encoder_trainer = Trainer(
    model=encoder_model, args=encoder_args,
    train_dataset=encoder_ds['train'],
    eval_dataset=encoder_ds['validation'],
    data_collator=DataCollatorWithPadding(encoder_tokenizer),
)

# actually train the model based on the parameters
encoder_trainer.train()


The model now predicts which class (action, route) that any input text belongs to. We'll test it on the test set to see how it does.

In [ ]:
# get predictions
encoder_logits = encoder_trainer.predict(encoder_ds['test']).predictions
encoder_predictions = encoder_logits.argmax(axis=-1)

# assemble metrics
encoder_metrics = {
    "model": "DistilBERT (encoder)",
    "accuracy": accuracy_score(encoder_ds['test']["label"], encoder_predictions),
    "macro F1": f1_score(encoder_ds['test']["label"], encoder_predictions, average="macro"),
}

# view metrics
pd.DataFrame(encoder_metrics, index=[0])

### B. Fine-tune the decoder-only model

We'll do the same thing here, with a few changes relative to the padding token and using the KV cache, to fine-tune our decoder model


In [ ]:
# get tokenizer and set padding token
decoder_name = "distilbert/distilgpt2"
decoder_tokenizer = AutoTokenizer.from_pretrained(decoder_name)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

# prepare the text inputs and labels with the tokenizer
decoder_ds = ds.map(lambda batch: tokenize(decoder_tokenizer, batch), batched=True)

# setup classification head, removing DistilGPT CLM head first
decoder_model = AutoModelForSequenceClassification.from_pretrained(
    decoder_name, num_labels=num_labels, id2label=id2label,
    label2id=label2id
)

# a bit of additional set up for the decoder model
decoder_model.config.pad_token_id = decoder_tokenizer.pad_token_id
decoder_model.config.use_cache = False  # Classification does not use autoregressive caching.

# prepare the training arguments
decoder_args = TrainingArguments(output_dir="/content/router-decoder",
                                 **shared_training_arguments, #comment
                                 #**{**shared_training_arguments, 'num_train_epochs':4}, #uncomment
                                 )

# setup the infrastructure for training
decoder_trainer = Trainer(
    model=decoder_model, args=decoder_args,
    train_dataset=decoder_ds['train'],
    eval_dataset=decoder_ds['validation'],
    data_collator=DataCollatorWithPadding(decoder_tokenizer),)

# actually train the model based on the parameters
decoder_trainer.train()

In [ ]:
# get predictions
decoder_logits = decoder_trainer.predict(decoder_ds['test']).predictions
decoder_predictions = decoder_logits.argmax(axis=-1)

# assemble metrics
decoder_metrics = {
    "model": "DistilGPT2 (decoder)",
    "accuracy": accuracy_score(decoder_ds['test']["label"], decoder_predictions),
    "macro F1": f1_score(decoder_ds['test']["label"], decoder_predictions, average="macro"),
}

# view metrics
pd.DataFrame([encoder_metrics, decoder_metrics])

### Compare the results
Try to train the decoder until you get to about the same loss as the encoder, compared as shown in the cell above. You can do this by going back up to the cell that trains the decoder, comment out line 22, and then remove the comment from line 23 so that it runs. Change the number of epochs so that the decoder model has a chance to train for more epochs.

**How much training does it take to get similar classification performance? Speculate.**

**Companion document: you're now ready to answer questions in the Transfer Learning section.** Consider both the observed results and why this is not a fully controlled comparison of attention masks alone.

In [ ]:
# free up some cuda memory
import gc

del decoder_trainer, decoder_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 2. Batches, padding, and model-output dimensions

So far, we've explored how transformers process sequences of tokens. **Batching** allows us to process *multiple* sequences together, improving computational efficiency and making better use of GPU hardware.

<img
  src="https://github.com/csbell-vu/attention-exploration/blob/main/tl-assets/bell-6340-hidden-dimension.png?raw=1"
  align="right"
  width="60%"
  hspace="20"
  wspace="10"
/>

In other words, we've only been thinking about 1 sequence of several tokens, e.g.,

> ["The cat is cute"]

but batching allows us to process several independent sequences at the same time:

> [  
>   &nbsp;&nbsp;&nbsp;["The cat is cute"],  
>   &nbsp;&nbsp;&nbsp;["And so is the dog"],  
>    &nbsp;&nbsp;&nbsp;["And the dog is also the baby."]  
> ]

Each element of a batch is an independent sequence of tokens, and the model performs the same operations on all sequences in parallel.

### Representation

Within each sequence, every token is represented by a vector of features (its embedding or hidden state) as we have seen in class. This gives us a three-dimensional tensor with shape:

> (`batch size`, `sequence length`, `hidden dimension`)

Although the sequences are processed together, they do not attend to one another in terms of attention. **Batching generally improves overall throughput, rather than necessarily making an individual prediction faster.** We can see what a batch looks like above, which focuses on demonstrating the alignment of the last dimension (hidden/representation).


### Padding

<img
  src="https://github.com/csbell-vu/attention-exploration/blob/main/tl-assets/bell-6340-padding-dimension.png?raw=1"
  align="right"
  width="50%"
  hspace="30"
  wspace="10"
/>

The <b>number of tokens</b> in each sequence must be identical. When sequences have uneven token lengths, we add <b>padding</b> tokens to make all the sequences the same length.

The figure on the right shows a focus on the first two dimensions of the examples, which is 3 elements in the batch, and the number of tokens in each sequence. Sequence 1 has 2 tokens, Sequence 3 has 5 tokens, and Sequence 3 has 4 tokens. Without the shown padding tokens, we would call this a **ragged array** because the sequences have different lengths.

To make the sequences equal length for computation, we pad each sequence (as needed) to some length. In this example, we pad to the <b>length of the longest token sequence in the batch</b>, though we could also <b>pad</b> to the maximum context width of the model. Look carefully. The first sequence has length 2, the second has length 5, and the the third has length 4. We add padding tokens to the shorter token lengths to make all sequences the same length. **All elements in a dimension must be of equal length.**

We can see this behavior below.


In [ ]:
# Create tokenizer and language model
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
language_model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2").to(device).eval()

In [ ]:
# make texts of different lengths
texts = [
    "The professor opened the notebook.",
    "In the next training step, the model predicts tokens in every sequence.",
    "Hello!",
    "A longer example can force shorter examples to receive more padding tokens.",
]

# tokenize these different length texts to the same number of tokens
batch = tokenizer(texts, padding=True, truncation=True, max_length=32, return_tensors="pt")

# print shape of input IDs
print(f"Tokenized dimensions: {batch["input_ids"].shape}\n")

# look at input IDs from the batch
batch["input_ids"]


Now, let's look at the output shape of the batch once we've run it through the model.

In [ ]:
with torch.inference_mode():
    outputs = language_model(**batch.to(device))

batch["input_ids"].shape, outputs.logits.shape, language_model.config.vocab_size

**Companion document: you're now ready to answer the questions in the Batching and Padding section.** Think about the batch dimension, sequence dimension, and vocabulary dimension.


## 3. Does the KV cache speed up autoregressive inference?

We'll do an experiment to explore this by:
* Dividing generation up into prefill vs decoding and measuring the duration of each, and
* Exploring the effect of the KV cache during generation as the sequence length increases.

Most of the code here is truly just infrastructure code unrelated to insights into transformers. Feel free to just execute.

**Note that because we're loading another model into the GPU here, you may need to restart your session, install the transformers package, and then you can scroll back down here to begin execution.**


In [ ]:
import torch
import time
import statistics
import matplotlib.pyplot as plt

In [ ]:
# load model into GPU
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M").to(device)
model.pad_token_id = model.eos_token_id
model.eval();

Below is a function that will basically run whatever function (`fn`) `repeats` number of times. You can pass in the function.

In [ ]:
def benchmark(fn, repeats=3):
    times = []

    with torch.inference_mode():
        fn()  # Warmup

        for _ in range(repeats):
            torch.cuda.synchronize()
            start = time.perf_counter()

            fn()

            torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)

    return statistics.median(times)

For a meaningful timing comparison, we use the **same prompt**, **same number of prompt tokens**, and **greedy decoding**. These small-scale timings can be noisy: caching has bookkeeping overhead, and its benefit tends to become more pronounced as sequences lengthen.

In [ ]:
# Experiment settings
generation_lengths = [16, 64, 128, 256, 512]
base_tokens = 32
vocab_size = model.get_input_embeddings().num_embeddings

# Generate random sequence for input prompt
prompt = {
    "input_ids": torch.randint(0, vocab_size, (1, base_tokens), device="cuda"),
    "attention_mask": torch.ones((1, base_tokens), device="cuda", dtype=torch.long)
}

# For results storage
results = []

Now, let's begin our experimentation for our generation lengths.

In [ ]:
for n in generation_lengths:

    # Prefill only
    prefill = benchmark(
        lambda: model(**prompt, use_cache=False)
    )

    # Function to generate tokens of length n
    def generate(use_cache):
        return model.generate(
            **prompt,
            max_new_tokens=n,
            min_new_tokens=n,
            do_sample=False,
            use_cache=use_cache
        )

    # Run the experiments for the generation length n
    cached = benchmark(lambda: generate(True))
    uncached = benchmark(lambda: generate(False))

    # Save this result run
    results.append((n, prefill, cached, uncached))

    # Print a lil something
    print(f"{n:4d} tokens | Prefill: {prefill:7.1f} ms | "
          f"Cached: {cached:7.1f} ms | "
          f"Uncached: {uncached:7.1f} ms | "
          f"Speedup: {uncached/cached:.2f}x")

Now, let's show what this looks like.


In [ ]:
# create dataframe
df = pd.DataFrame(results, columns=["Length", "Prefill", "Cached", "Uncached"])
df.set_index('Length')

# plot
df.plot(kind='line', x='Length', style="-o", ylabel="Time (ms)");

What do you notice about using the KV cache versus not using it for generation? How does this compare to the time required to compute the prefix? Note that we did not include the setup time for the cache in these calculations, which does have an impact as well.

**Companion document:** You're now ready to answer questions in  the KV Cache section!
